# Fase 0 — Auditoria e qualidade do dataset CorrDiff

Este notebook **não executa a varredura pesada do Zarr**. A varredura é realizada pelo script `scripts/00_audit_dataset.py`, que lê o dataset em lotes e grava resultados reduzidos em `analysis_outputs/00_quality/`.

O objetivo desta fase é responder, antes de qualquer análise meteorológica ou treinamento:

- a estrutura do Zarr está consistente?
- a ordem e o número de canais estão corretos?
- qual é a cobertura temporal real entre 2011 e 2024?
- existem anos, meses ou horários com lacunas sistemáticas?
- quantos patches são produzidos por timestamp?
- a geometria `patch_size=32`, `stride=16` cobre toda a grade 70×84?
- existem NaN/Inf nos inputs gravados?
- a máscara é realmente binária?
- os valores do target são coerentes com a máscara?
- as estatísticas recalculadas são compatíveis com as estatísticas salvas pelo builder?

> **Nota científica importante:** o builder atual registra `target_transform = log1p`. Antes de interpretar o target em termos físicos ou usar limiares em dBZ, é necessário confirmar a escala física da refletividade de origem e revisar se essa transformação é adequada.

## 0. Execução do auditor

A execução completa recomendada, a partir da raiz do projeto, é:

```bash
python scripts/00_audit_dataset.py \
  --dataset-dir datasets/corrdiff_2011_2024 \
  --output-dir analysis_outputs/00_quality \
  --batch-samples 512 \
  --quantile-sample-values 200000 \
  --overwrite
```

Para validar rapidamente caminhos e estrutura sem percorrer todos os pixels:

```bash
python scripts/00_audit_dataset.py \
  --dataset-dir datasets/corrdiff_2011_2024 \
  --output-dir analysis_outputs/00_quality_quick \
  --quick \
  --overwrite
```

A execução `full` calcula contagens de NaN/Inf, média, desvio padrão, mínimo e máximo de forma exata sobre o Zarr. Os quantis são aproximados por amostragem controlada para evitar carregar dezenas de gigabytes em memória.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

# Se o JupyterLab foi iniciado na raiz do projeto, os defaults abaixo funcionam.
# Você também pode definir CORRDIFF_DATASET_DIR e CORRDIFF_QUALITY_DIR no shell.
DATASET_DIR = Path(os.environ.get("CORRDIFF_DATASET_DIR", "datasets/corrdiff_2011_2024"))
QUALITY_DIR = Path(os.environ.get("CORRDIFF_QUALITY_DIR", "analysis_outputs/00_quality"))

print("DATASET_DIR:", DATASET_DIR.resolve())
print("QUALITY_DIR:", QUALITY_DIR.resolve())

In [ ]:
required_files = [
    QUALITY_DIR / "dataset_summary.json",
    QUALITY_DIR / "audit_warnings.json",
    QUALITY_DIR / "yearly_coverage.parquet",
    QUALITY_DIR / "monthly_coverage.parquet",
    QUALITY_DIR / "hourly_coverage.parquet",
    QUALITY_DIR / "temporal_coverage.parquet",
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Execute scripts/00_audit_dataset.py antes deste notebook. Arquivos ausentes:\n- "
        + "\n- ".join(missing)
    )

with (QUALITY_DIR / "dataset_summary.json").open("r", encoding="utf-8") as f:
    summary = json.load(f)

with (QUALITY_DIR / "audit_warnings.json").open("r", encoding="utf-8") as f:
    audit_messages = json.load(f)

yearly = pd.read_parquet(QUALITY_DIR / "yearly_coverage.parquet")
monthly = pd.read_parquet(QUALITY_DIR / "monthly_coverage.parquet")
hourly = pd.read_parquet(QUALITY_DIR / "hourly_coverage.parquet")
temporal = pd.read_parquet(QUALITY_DIR / "temporal_coverage.parquet")

channel_path = QUALITY_DIR / "channel_summary.parquet"
missing_path = QUALITY_DIR / "missing_data.parquet"
patch_position_path = QUALITY_DIR / "patch_position_counts.parquet"

channel_summary = pd.read_parquet(channel_path) if channel_path.exists() else pd.DataFrame()
missing_data = pd.read_parquet(missing_path) if missing_path.exists() else pd.DataFrame()
patch_positions = pd.read_parquet(patch_position_path) if patch_position_path.exists() else pd.DataFrame()

print("Status do audit:", summary.get("status"))
print("Modo:", summary.get("audit_mode"))
print("Tempo de execução (s):", round(summary.get("elapsed_seconds", 0), 2))

## 1. Resumo estrutural

In [ ]:
dataset = summary["dataset"]
config = summary["configuration"]
temporal_summary = summary["temporal"]
patch_geometry = summary["patch_geometry"]

structural = pd.DataFrame(
    [
        ("Amostras (patches)", dataset.get("num_samples")),
        ("Canais de entrada", dataset.get("num_channels")),
        ("Shape input", str(dataset.get("input_shape"))),
        ("Shape target", str(dataset.get("target_shape"))),
        ("Shape mask", str(dataset.get("mask_shape"))),
        ("Dimensão amostral consistente", dataset.get("sample_dimension_consistent")),
        ("Transformação do target", dataset.get("target_transform")),
        ("Período configurado", f"{config.get('start_date')} → {config.get('end_date')}"),
        ("Frequência temporal", config.get("time_frequency")),
        ("Grade do radar", str(config.get("radar_grid_shape"))),
        ("Resolução do radar (km)", config.get("radar_resolution_km")),
        ("Patch size", config.get("patch_size")),
        ("Stride", config.get("stride")),
    ],
    columns=["Item", "Valor"],
)

display(structural)

In [ ]:
channels = pd.DataFrame(
    {
        "índice": range(len(dataset.get("channels", []))),
        "canal": dataset.get("channels", []),
    }
)
display(channels)

### Configuração esperada neste experimento

A configuração científica planejada é:

- **superfície:** `tcwv`, `t2m`, `u10`, `v10`;
- **850 hPa:** `t_850`, `r_850`, `u_850`, `v_850`;
- **500 hPa:** `t_500`, `r_500`, `u_500`, `v_500`;
- **12 canais ERA5** ao todo;
- **radar:** grade 70×84, resolução espacial de 2 km;
- **dataset:** frequência horária;
- **patches:** 32×32, stride 16.

A tabela acima é a fonte efetiva do Zarr/metadata e deve prevalecer sobre qualquer expectativa teórica.

## 2. Avisos e notas do auditor

In [ ]:
warnings = audit_messages.get("warnings", [])
notes = audit_messages.get("notes", [])

if warnings:
    display(Markdown("### ⚠️ Avisos"))
    for item in warnings:
        display(Markdown(f"- **{item['code']}** — {item['message']}"))
else:
    display(Markdown("✅ **Nenhum aviso estrutural crítico foi produzido pelo auditor.**"))

if notes:
    display(Markdown("### Notas"))
    for item in notes:
        display(Markdown(f"- **{item['code']}** — {item['message']}"))

## 3. Cobertura temporal global

In [ ]:
temporal_table = pd.DataFrame(
    [
        ("Timestamps esperados", temporal_summary.get("expected_timestamps")),
        ("Timestamps com pelo menos um patch", temporal_summary.get("available_timestamps")),
        ("Timestamps sem patch", temporal_summary.get("missing_timestamps")),
        ("Cobertura temporal", temporal_summary.get("coverage_ratio")),
        ("Primeiro timestamp observado", temporal_summary.get("first_observed_timestamp")),
        ("Último timestamp observado", temporal_summary.get("last_observed_timestamp")),
        ("Unidade detectada no Zarr", temporal_summary.get("timestamp_storage_unit_detected")),
    ],
    columns=["Métrica", "Valor"],
)
display(temporal_table)

coverage = temporal_summary.get("coverage_ratio")
if coverage is not None:
    print(f"Cobertura global: {coverage:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(yearly["year"].astype(str), yearly["coverage_ratio"] * 100)
ax.set_ylim(0, 100)
ax.set_ylabel("Cobertura temporal (%)")
ax.set_xlabel("Ano")
ax.set_title("Cobertura temporal do dataset por ano")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(yearly)

**Interpretação:** quedas localizadas em um ano não devem ser interpretadas inicialmente como sinal meteorológico. Primeiro devem ser confrontadas com os contadores do builder (`missing_radar`, `missing_era5`, patches rejeitados) e com a disponibilidade do arquivo histórico do radar.

## 4. Cobertura mensal — procura por lacunas sistemáticas

In [ ]:
coverage_matrix = monthly.pivot(index="year", columns="month", values="coverage_ratio") * 100
coverage_matrix = coverage_matrix.reindex(columns=range(1, 13))

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(coverage_matrix.values, aspect="auto", vmin=0, vmax=100)
ax.set_title("Cobertura temporal por ano e mês (%)")
ax.set_xlabel("Mês")
ax.set_ylabel("Ano")
ax.set_xticks(range(12), range(1, 13))
ax.set_yticks(range(len(coverage_matrix.index)), coverage_matrix.index)
fig.colorbar(im, ax=ax, label="Cobertura (%)")
plt.tight_layout()
plt.show()

In [ ]:
worst_months = monthly.sort_values(["coverage_ratio", "year", "month"]).head(24)
display(worst_months)

## 5. Cobertura por hora UTC

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(hourly["hour_utc"], hourly["coverage_ratio"] * 100)
ax.set_xticks(range(24))
ax.set_ylim(0, 100)
ax.set_xlabel("Hora UTC")
ax.set_ylabel("Cobertura (%)")
ax.set_title("Cobertura de timestamps por hora UTC")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

display(hourly)

**Por que isso importa:** uma cobertura muito diferente entre horas pode enviesar a análise posterior do ciclo diurno. Antes de concluir que determinada hora possui mais ou menos precipitação, precisamos saber se ela possui a mesma disponibilidade observacional.

## 6. Quantidade de patches por timestamp

In [ ]:
patch_stats = temporal_summary.get("patches_per_timestamp", {})
display(pd.DataFrame([patch_stats]))

available_temporal = temporal.loc[temporal["available"]].copy()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(available_temporal["samples"], bins=np.arange(0.5, available_temporal["samples"].max() + 1.5, 1))
ax.set_xlabel("Patches gravados no timestamp")
ax.set_ylabel("Número de timestamps")
ax.set_title("Distribuição de patches válidos por timestamp")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

Para a grade 70×84 com `patch_size=32` e `stride=16`, a geometria convencional do builder produz no máximo **12 âncoras** por campo completo (3 posições verticais × 4 horizontais), antes dos filtros de validade.

## 7. Geometria espacial dos patches

In [ ]:
geometry_table = pd.DataFrame(
    [
        ("Âncoras esperadas por campo", patch_geometry.get("expected_patch_positions_per_full_field")),
        ("Âncoras observadas", patch_geometry.get("actual_unique_patch_positions")),
        ("Cobertura espacial esperada", patch_geometry.get("expected_spatial_coverage_ratio")),
        ("Cobertura espacial observada", patch_geometry.get("actual_spatial_coverage_ratio")),
    ],
    columns=["Métrica", "Valor"],
)
display(geometry_table)

if not patch_positions.empty:
    pivot = patch_positions.pivot(index="patch_row", columns="patch_col", values="samples").fillna(0)
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)), pivot.columns)
    ax.set_yticks(range(len(pivot.index)), pivot.index)
    ax.set_xlabel("patch_col")
    ax.set_ylabel("patch_row")
    ax.set_title("Número de amostras por posição de patch")
    fig.colorbar(im, ax=ax, label="Amostras")
    plt.tight_layout()
    plt.show()
    display(patch_positions)

In [ ]:
coverage_file = QUALITY_DIR / "expected_patch_spatial_coverage.npy"
if coverage_file.exists():
    patch_coverage = np.load(coverage_file)
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(patch_coverage, origin="upper")
    ax.set_title("Número de patches que cobrem cada pixel da grade 70×84")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax, label="Número de sobreposições")
    plt.tight_layout()
    plt.show()

    uncovered = np.argwhere(patch_coverage == 0)
    print("Pixels sem cobertura de patch:", len(uncovered))
    print("Cobertura espacial:", f"{(patch_coverage > 0).mean():.2%}")

### Observação metodológica sobre as bordas

Com o loop atual baseado em `range(0, tamanho - patch_size + 1, stride)`, uma dimensão cujo restante não seja compatível com o stride pode deixar pixels de borda sem cobertura. Isso **não invalida** o dataset, mas precisa ser conhecido ao reconstruir campos completos ou calcular climatologias espaciais a partir dos patches.

Uma alternativa futura seria adicionar explicitamente a última âncora `tamanho - patch_size` para garantir cobertura integral, caso isso seja desejável no desenho experimental.

## 8. Qualidade numérica dos canais

In [ ]:
if channel_summary.empty:
    display(Markdown("O auditor foi executado em modo `quick` sem estatísticas numéricas completas."))
else:
    cols = [
        c for c in [
            "kind", "channel_index", "channel", "finite_ratio", "mean", "std", "min", "max",
            "p01", "p05", "p25", "p50", "p75", "p95", "p99",
            "quantile_sample_count", "mean_abs_diff_vs_builder", "std_abs_diff_vs_builder"
        ] if c in channel_summary.columns
    ]
    display(channel_summary[cols])

In [ ]:
if not missing_data.empty:
    display(missing_data)

    nonfinite = missing_data.copy()
    nonfinite["nonfinite_count"] = (
        nonfinite["nan_count"] + nonfinite["posinf_count"] + nonfinite["neginf_count"]
    )
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.barh(nonfinite["channel"], nonfinite["nonfinite_count"])
    ax.set_xlabel("NaN + Inf")
    ax.set_title("Valores não finitos por canal/array")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

No dataset atual, `minimum_input_valid_ratio=1.0` implica que patches com qualquer insuficiência de validade no tensor de entrada deveriam ser rejeitados antes da escrita. Portanto, encontrar NaN/Inf no `input` gravado merece investigação.

## 9. Máscara e integridade do target

In [ ]:
mask_info = summary.get("mask", {})
target_info = summary.get("target_integrity", {})

display(pd.DataFrame([mask_info]))
display(pd.DataFrame([target_info]))

if mask_info:
    print("Proporção global de pixels válidos do radar:", f"{mask_info.get('valid_pixel_ratio', float('nan')):.2%}")

### Ponto que deve ser resolvido antes das simulações

O builder atual preenche pixels inválidos do radar com zero, aplica `clip(..., 0, None)` e depois `log1p`. Se a fonte original do radar estiver realmente em **dBZ**, isso merece revisão científica, porque dBZ já é uma escala logarítmica e valores negativos de dBZ podem ter significado físico.

Nesta fase, não alteramos o dataset. Apenas registramos a transformação para que a decisão seja tomada antes da interpretação de extremos e do treinamento final.

## 10. Contadores do builder

In [ ]:
builder_counters = summary.get("builder_counters", {})
if builder_counters:
    counters_df = pd.DataFrame(
        sorted(builder_counters.items()), columns=["contador", "valor"]
    )
    display(counters_df)
else:
    print("metadata.json não contém counters do builder.")

## 11. Diagnóstico automático para avançar à Fase 1

In [ ]:
checks = []

def add_check(name, ok, detail):
    checks.append({"checagem": name, "status": "OK" if ok else "REVISAR", "detalhe": detail})

add_check(
    "Dimensões do Zarr",
    bool(dataset.get("sample_dimension_consistent")),
    str(dataset.get("sample_dimensions")),
)
add_check(
    "12 canais ERA5",
    dataset.get("num_channels") == 12,
    f"C={dataset.get('num_channels')} | {dataset.get('channels')}",
)
add_check(
    "Grade do radar 70×84",
    config.get("radar_grid_shape") == [70, 84],
    str(config.get("radar_grid_shape")),
)
add_check(
    "Patch 32 / stride 16",
    config.get("patch_size") == 32 and config.get("stride") == 16,
    f"patch={config.get('patch_size')}, stride={config.get('stride')}",
)
add_check(
    "Cobertura temporal conhecida",
    temporal_summary.get("coverage_ratio") is not None,
    f"coverage={temporal_summary.get('coverage_ratio')}",
)

if not missing_data.empty:
    input_rows = missing_data.loc[missing_data["kind"] == "input"]
    input_nonfinite = int(
        input_rows[["nan_count", "posinf_count", "neginf_count"]].sum().sum()
    )
    add_check("Inputs sem NaN/Inf", input_nonfinite == 0, f"nonfinite={input_nonfinite}")

if summary.get("mask"):
    add_check(
        "Máscara binária",
        summary["mask"].get("binary_violation_count", 0) == 0,
        f"violations={summary['mask'].get('binary_violation_count')}",
    )

checks_df = pd.DataFrame(checks)
display(checks_df)

if (checks_df["status"] == "REVISAR").any() or summary.get("warnings_count", 0) > 0:
    display(Markdown("### Resultado: **há itens a revisar antes da Fase 1**"))
else:
    display(Markdown("### Resultado: **dataset apto a avançar para a Fase 1 — análise univariada**"))

## 12. O que levamos para a próxima fase

Ao concluir esta auditoria, registre no texto da dissertação:

1. período temporal efetivamente disponível;
2. número de timestamps e patches válidos;
3. cobertura por ano/mês/hora;
4. ordem exata dos 12 canais;
5. resolução espacial e temporal;
6. geometria e cobertura dos patches;
7. percentual de pixels válidos do radar;
8. presença/ausência de valores não finitos;
9. transformação efetivamente aplicada ao target;
10. qualquer período ou posição espacial sub-representada.

A **Fase 1** deverá então estudar as distribuições univariadas em maior detalhe: histogramas, ECDF, quantis, assimetria, curtose e dependência sazonal das distribuições, sempre condicionando a interpretação do radar à decisão sobre sua escala física.